# Raw to Silver - Biblioteca
Lee los CSVs de la capa **raw** en ADLS Gen2, detecta problemas de calidad,
limpia los datos y los persiste en formato **Delta** en la capa **silver**.

## 0. Configuracion del entorno local

Esta celda solo se ejecuta **fuera de Databricks** (ejecucion local).
Configura PySpark con acceso a ADLS Gen2 usando la storage account key
definida en el fichero `.env` de la raiz del proyecto.

En Databricks esta celda no tiene efecto: el cluster ya tiene
SparkSession y autenticacion configurados.

In [1]:
import os
import importlib

is_local_environment = not os.environ.get("DATABRICKS_RUNTIME_VERSION")

if is_local_environment:
    REQUIRED_PACKAGES = {"pyspark": "pyspark", "dotenv": "python-dotenv", "delta": "delta-spark"}
    missing_packages = [
        pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
        if not importlib.util.find_spec(module_name)
    ]

    if missing_packages:
        %pip install {' '.join(missing_packages)} --quiet

    from dotenv import load_dotenv
    from pyspark.sql import SparkSession

    load_dotenv()

    spark = (
        SparkSession.builder
        .appName("local-notebook")
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-azure:3.4.2,"
            "io.delta:delta-spark_2.13:4.1.0",
        )
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

    storage_account_key = os.environ["ADLS_KEY"]
    spark.conf.set("fs.azure.account.key.stadatademo2026.dfs.core.windows.net", storage_account_key)

    print("[LOCAL] SparkSession configurada con acceso a ADLS")

[LOCAL] SparkSession configurada con acceso a ADLS


In [2]:
STORAGE_ACCOUNT_NAME = "stadatademo2026"
RAW_CONTAINER = "raw"
SILVER_CONTAINER = "silver"

RAW_BASE_PATH = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"
SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"

TABLE_NAMES = ["autores", "libros", "usuarios", "prestamos"]

## 1. Lectura de CSVs desde la capa raw

In [3]:
from pyspark.sql import DataFrame

raw_dataframes: dict[str, DataFrame] = {}

for table_name in TABLE_NAMES:
    raw_path = f"{RAW_BASE_PATH}/{table_name}.csv"
    raw_dataframes[table_name] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_path)
    )

for table_name, dataframe in raw_dataframes.items():
    print(f"\n{'='*60}")
    print(f"  {table_name.upper()} — {dataframe.count()} filas, {len(dataframe.columns)} columnas")
    print(f"{'='*60}")
    dataframe.printSchema()
    dataframe.show(5, truncate=False)


  AUTORES — 150 filas, 4 columnas
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- nacionalidad: string (nullable = true)
 |-- fecha_nacimiento: date (nullable = true)

+---+----------------------+------------+----------------+
|id |nombre                |nacionalidad|fecha_nacimiento|
+---+----------------------+------------+----------------+
|1  |Albano Llopis Hierro  |Guatemala   |1978-10-13      |
|2  |Rafa Calatayud Bonet  |Mexico      |1950-06-16      |
|3  |Alejandra Andrés Bueno|España      |1975-06-14      |
|4  |José Antonio de Flor  |Honduras    |1950-02-23      |
|5  |Hortensia de Sanchez  |Chile       |1983-12-29      |
+---+----------------------+------------+----------------+
only showing top 5 rows

  LIBROS — 400 filas, 7 columnas
root
 |-- id: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- isbn: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- fecha_publicacion: date (nullable = true)
 |-- edi

## 2. Deteccion de problemas de calidad

In [4]:
from pyspark.sql import functions as F

quality_log_entries: list[dict] = []


def _append_quality_entry(table: str, check_name: str, affected_row_count: int, detail: str):
    quality_log_entries.append({
        "table": table,
        "check": check_name,
        "affected_rows": affected_row_count,
        "detail": detail,
    })


def _detect_duplicate_rows(table_name: str, dataframe: DataFrame):
    total_row_count = dataframe.count()
    distinct_row_count = dataframe.distinct().count()
    duplicate_count = total_row_count - distinct_row_count

    _append_quality_entry(
        table_name,
        "duplicados_exactos",
        duplicate_count,
        f"{duplicate_count} filas duplicadas de {total_row_count} totales",
    )


def _detect_null_values_per_column(table_name: str, dataframe: DataFrame):
    for column_name in dataframe.columns:
        null_count = dataframe.filter(F.col(column_name).isNull()).count()
        has_null_values = null_count > 0
        if has_null_values:
            _append_quality_entry(
                table_name,
                f"nulos_{column_name}",
                null_count,
                f"Columna '{column_name}' tiene {null_count} valores nulos",
            )


def _detect_inconsistent_genre_casing(dataframe: DataFrame):
    distinct_genre_rows = dataframe.select("genero").distinct().collect()
    distinct_genre_values = [row["genero"] for row in distinct_genre_rows]
    normalized_genres = set(value.strip().lower() for value in distinct_genre_values if value)

    has_casing_inconsistency = len(distinct_genre_values) > len(normalized_genres)
    if has_casing_inconsistency:
        _append_quality_entry(
            "libros",
            "genero_casing_inconsistente",
            len(distinct_genre_values) - len(normalized_genres),
            f"Valores distintos encontrados: {sorted(distinct_genre_values)}",
        )


def _detect_returned_loans_without_return_date(dataframe: DataFrame):
    is_returned = F.col("estado") == "devuelto"
    has_no_return_date = F.col("fecha_devolucion").isNull()
    returned_without_date_count = dataframe.filter(is_returned & has_no_return_date).count()

    has_inconsistency = returned_without_date_count > 0
    if has_inconsistency:
        _append_quality_entry(
            "prestamos",
            "devuelto_sin_fecha_devolucion",
            returned_without_date_count,
            f"{returned_without_date_count} prestamos con estado 'devuelto' pero sin fecha_devolucion",
        )


def _detect_semantic_duplicates_in_loans(dataframe: DataFrame):
    loan_identity_columns = ["libro_id", "usuario_id", "fecha_prestamo"]
    grouped_by_identity = dataframe.groupBy(loan_identity_columns).count()
    semantic_duplicates = grouped_by_identity.filter(F.col("count") > 1)
    semantic_duplicate_count = semantic_duplicates.count()

    has_semantic_duplicates = semantic_duplicate_count > 0
    if has_semantic_duplicates:
        total_duplicated_rows = semantic_duplicates.agg(F.sum("count")).collect()[0][0]
        extra_rows = total_duplicated_rows - semantic_duplicate_count
        _append_quality_entry(
            "prestamos",
            "duplicados_semanticos",
            int(extra_rows),
            f"{semantic_duplicate_count} combinaciones libro+usuario+fecha repetidas ({int(extra_rows)} filas extra)",
        )


for table_name, dataframe in raw_dataframes.items():
    _detect_duplicate_rows(table_name, dataframe)
    _detect_null_values_per_column(table_name, dataframe)

_detect_inconsistent_genre_casing(raw_dataframes["libros"])
_detect_returned_loans_without_return_date(raw_dataframes["prestamos"])
_detect_semantic_duplicates_in_loans(raw_dataframes["prestamos"])

print("Problemas de calidad detectados:")
print("-" * 80)
for entry in quality_log_entries:
    has_affected_rows = entry["affected_rows"] > 0
    status_icon = "ISSUE" if has_affected_rows else "OK"
    print(f"[{status_icon}] {entry['table']}.{entry['check']}: {entry['detail']}")

Problemas de calidad detectados:
--------------------------------------------------------------------------------
[OK] autores.duplicados_exactos: 0 filas duplicadas de 150 totales
[OK] libros.duplicados_exactos: 0 filas duplicadas de 400 totales
[OK] usuarios.duplicados_exactos: 0 filas duplicadas de 250 totales
[OK] prestamos.duplicados_exactos: 0 filas duplicadas de 498 totales
[ISSUE] prestamos.nulos_fecha_devolucion: Columna 'fecha_devolucion' tiene 31 valores nulos
[ISSUE] libros.genero_casing_inconsistente: Valores distintos encontrados: ['BIOGRAFIA', 'Biografia', 'CIENCIA FICCION', 'Ciencia Ficcion', 'ENSAYO', 'Ensayo', 'HISTORIA', 'Historia', 'NOVELA', 'Novela', 'POESIA', 'Poesia', 'biografia', 'ciencia ficcion', 'ensayo', 'historia', 'novela', 'poesia']
[ISSUE] prestamos.devuelto_sin_fecha_devolucion: 31 prestamos con estado 'devuelto' pero sin fecha_devolucion
[ISSUE] prestamos.duplicados_semanticos: 23 combinaciones libro+usuario+fecha repetidas (23 filas extra)


## 3. Limpieza y transformacion a Silver

In [6]:
from pyspark.sql.types import DateType, IntegerType


def _remove_exact_duplicates(dataframe: DataFrame) -> DataFrame:
    return dataframe.dropDuplicates()


def _cast_date_columns(dataframe: DataFrame, date_column_names: list[str]) -> DataFrame:
    result = dataframe
    for column_name in date_column_names:
        is_column_present = column_name in result.columns
        if is_column_present:
            result = result.withColumn(column_name, F.col(column_name).cast(DateType()))
    return result


def _cast_id_columns_to_integer(dataframe: DataFrame) -> DataFrame:
    result = dataframe
    for column_name in result.columns:
        is_id_column = column_name == "id" or column_name.endswith("_id")
        if is_id_column:
            result = result.withColumn(column_name, F.col(column_name).cast(IntegerType()))
    return result


def _normalize_genre_casing(dataframe: DataFrame) -> DataFrame:
    return dataframe.withColumn("genero", F.lower(F.trim(F.col("genero"))))


def _remove_semantic_duplicate_loans(dataframe: DataFrame) -> DataFrame:
    from pyspark.sql.window import Window

    loan_identity_columns = ["libro_id", "usuario_id", "fecha_prestamo"]
    window_by_loan_identity = Window.partitionBy(loan_identity_columns).orderBy("id")
    with_row_number = dataframe.withColumn("_row_number", F.row_number().over(window_by_loan_identity))

    is_first_occurrence = F.col("_row_number") == 1
    return with_row_number.filter(is_first_occurrence).drop("_row_number")


def _fill_missing_return_dates_for_returned_loans(dataframe: DataFrame) -> DataFrame:
    is_returned = F.col("estado") == "devuelto"
    has_no_return_date = F.col("fecha_devolucion").isNull()
    needs_estimated_return_date = is_returned & has_no_return_date

    DEFAULT_LOAN_DURATION_DAYS = 14
    estimated_return_date = F.date_add(F.col("fecha_prestamo"), DEFAULT_LOAN_DURATION_DAYS)

    return dataframe.withColumn(
        "fecha_devolucion",
        F.when(needs_estimated_return_date, estimated_return_date).otherwise(F.col("fecha_devolucion")),
    )

In [7]:
def _clean_autores(dataframe: DataFrame) -> DataFrame:
    without_duplicates = _remove_exact_duplicates(dataframe)
    with_typed_ids = _cast_id_columns_to_integer(without_duplicates)
    with_typed_dates = _cast_date_columns(with_typed_ids, ["fecha_nacimiento"])
    return with_typed_dates.withColumn("nombre", F.trim(F.col("nombre")))


def _clean_libros(dataframe: DataFrame) -> DataFrame:
    without_duplicates = _remove_exact_duplicates(dataframe)
    with_typed_ids = _cast_id_columns_to_integer(without_duplicates)
    with_typed_dates = _cast_date_columns(with_typed_ids, ["fecha_publicacion"])
    with_normalized_genre = _normalize_genre_casing(with_typed_dates)
    return with_normalized_genre.withColumn("titulo", F.trim(F.col("titulo")))


def _clean_usuarios(dataframe: DataFrame) -> DataFrame:
    without_duplicates = _remove_exact_duplicates(dataframe)
    with_typed_ids = _cast_id_columns_to_integer(without_duplicates)
    with_typed_dates = _cast_date_columns(with_typed_ids, ["fecha_registro"])
    with_trimmed_name = with_typed_dates.withColumn("nombre", F.trim(F.col("nombre")))
    return with_trimmed_name.withColumn("tipo_socio", F.lower(F.trim(F.col("tipo_socio"))))


def _clean_prestamos(dataframe: DataFrame) -> DataFrame:
    without_duplicates = _remove_exact_duplicates(dataframe)
    with_typed_ids = _cast_id_columns_to_integer(without_duplicates)
    with_typed_dates = _cast_date_columns(with_typed_ids, ["fecha_prestamo", "fecha_devolucion"])
    without_semantic_duplicates = _remove_semantic_duplicate_loans(with_typed_dates)
    with_filled_return_dates = _fill_missing_return_dates_for_returned_loans(without_semantic_duplicates)
    return with_filled_return_dates.withColumn("estado", F.lower(F.trim(F.col("estado"))))


CLEANING_FUNCTIONS = {
    "autores": _clean_autores,
    "libros": _clean_libros,
    "usuarios": _clean_usuarios,
    "prestamos": _clean_prestamos,
}

silver_dataframes: dict[str, DataFrame] = {}

for table_name, raw_dataframe in raw_dataframes.items():
    cleaning_function = CLEANING_FUNCTIONS[table_name]
    silver_dataframes[table_name] = cleaning_function(raw_dataframe)

## 4. Validacion post-limpieza

In [8]:
print("Comparativa Raw vs Silver:")
print("=" * 80)
print(f"{'Tabla':<15} {'Raw':>10} {'Silver':>10} {'Eliminadas':>12}")
print("-" * 80)

for table_name in TABLE_NAMES:
    raw_count = raw_dataframes[table_name].count()
    silver_count = silver_dataframes[table_name].count()
    removed_count = raw_count - silver_count
    print(f"{table_name:<15} {raw_count:>10} {silver_count:>10} {removed_count:>12}")

print("\nSchema Silver:")
for table_name, dataframe in silver_dataframes.items():
    print(f"\n--- {table_name} ---")
    dataframe.printSchema()

Comparativa Raw vs Silver:
Tabla                  Raw     Silver   Eliminadas
--------------------------------------------------------------------------------
autores                150        150            0
libros                 400        400            0
usuarios               250        250            0
prestamos              498        475           23

Schema Silver:

--- autores ---
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- nacionalidad: string (nullable = true)
 |-- fecha_nacimiento: date (nullable = true)


--- libros ---
root
 |-- id: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- isbn: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- fecha_publicacion: date (nullable = true)
 |-- editorial: string (nullable = true)
 |-- autor_id: integer (nullable = true)


--- usuarios ---
root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- email: string (nullable = true)
 |--

## 5. Escritura en capa Silver (Delta)

In [10]:
for table_name, dataframe in silver_dataframes.items():
    silver_path = f"{SILVER_BASE_PATH}/{table_name}"
    dataframe.write.format("delta").mode("overwrite").save(silver_path)
    print(f"[OK] {table_name} -> {silver_path}")

Py4JJavaError: An error occurred while calling o404.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 186.0 failed 1 times, most recent failure: Lost task 0.0 in stage 186.0 (TID 116) (192.168.1.119 executor driver): java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.util.DiskChecker.checkAccessByFileMethods(DiskChecker.java:160)
	at org.apache.hadoop.util.DiskChecker.checkDirInternal(DiskChecker.java:100)
	at org.apache.hadoop.util.DiskChecker.checkDir(DiskChecker.java:77)
	at org.apache.hadoop.util.BasicDiskValidator.checkStatus(BasicDiskValidator.java:32)
	at org.apache.hadoop.fs.LocalDirAllocator$AllocatorPerContext.confChanged(LocalDi

## 6. Log de calidad

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType

QUALITY_LOG_SCHEMA = StructType([
    StructField("table", StringType(), False),
    StructField("check", StringType(), False),
    StructField("affected_rows", IntegerType(), False),
    StructField("detail", StringType(), False),
])

quality_log_rows = [
    (entry["table"], entry["check"], entry["affected_rows"], entry["detail"])
    for entry in quality_log_entries
]

quality_log_dataframe = spark.createDataFrame(quality_log_rows, schema=QUALITY_LOG_SCHEMA)
quality_log_dataframe = quality_log_dataframe.withColumn("execution_timestamp", F.current_timestamp())

quality_log_path = f"{SILVER_BASE_PATH}/_quality_log"
quality_log_dataframe.write.format("delta").mode("append").save(quality_log_path)

print(f"[OK] Log de calidad guardado en {quality_log_path}")
quality_log_dataframe.show(truncate=False)